We are performing Multiple OnFailActions on CompetitorCheck validator

In [1]:
from guardrails import Guard, OnFailAction
from guardrails.hub import CompetitorCheck
from guardrails.errors import ValidationError

/var/folders/03/0f3bjgwn6xg24ppkhkb89gzm0000gn/T/ipykernel_78064/1826108792.py:2: DeprecationWarning: Importing validators from `guardrails.hub` is deprecated and will be removed in a future major release. Import directly from the `guardrails_ai` namespace instead, e.g. `from guardrails_ai.<name> import CompetitorCheck`.
  from guardrails.hub import CompetitorCheck


In [2]:
from guardrails.hub import CompetitorCheck

COMPETITORS = ["Stripe", "PayPal", "Square"]
BAD_TEXT = (
    "Thanks for reaching out! Honestly, Stripe has lower fees than us, "
    "and PayPal is easier to set up. But we're happy to help with your refund."
)
GOOD_TEXT = (
    "Thanks for reaching out! We're happy to help with your refund."
)

In [3]:
def competitor_guard(action):
    return Guard().use(
        CompetitorCheck(competitors=COMPETITORS, on_fail=action, use_local=True)
    )

print("Input we'll validate five ways:\n", BAD_TEXT)

Input we'll validate five ways:
 Thanks for reaching out! Honestly, Stripe has lower fees than us, and PayPal is easier to set up. But we're happy to help with your refund.


### 1. Fix OnFailAction

In [ ]:
guard = competitor_guard(OnFailAction.FIX)
out = guard.validate(GOOD_TEXT)
print("Output Raw response", out)
print("validation_passed:", out.validation_passed)
print("OUTPUT:\n", out.validated_output)

/Users/rishabh013/prod_new/guardrails/gaurdrails_ai/.venv/lib/python3.14/site-packages/torch/jit/_script.py:1485: FutureWarning: `torch.jit.script` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/Users/rishabh013/prod_new/guardrails/gaurdrails_ai/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Output Raw response ValidationOutcome[TypeVar](
    call_id='4842919856',
    raw_llm_output="Thanks for reaching out! We're happy to help with your refund.",
    validation_summaries=[],
    validated_output="Thanks for reaching out! We're happy to help with your refund.",
    reask=None,
    validation_passed=True,
    error=None
)
validation_passed: True
OUTPUT:
 Thanks for reaching out! We're happy to help with your refund.


/Users/rishabh013/prod_new/guardrails/gaurdrails_ai/.venv/lib/python3.14/site-packages/guardrails/validator_service/__init__.py:81: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


ERROR:opentelemetry.exporter.otlp.proto.http.trace_exporter:Failed to export span batch due to timeout, max retries or shutdown.


In [ ]:
guard = competitor_guard(OnFailAction.FIX)
out = guard.validate(BAD_TEXT)
print("Output Raw response", out)
print("validation_passed:", out.validation_passed)
print("OUTPUT:\n", out.validated_output)

Output Raw response ValidationOutcome[TypeVar](
    call_id='4828041072',
    raw_llm_output="Thanks for reaching out! Honestly, Stripe has lower fees than us, and PayPal is easier to set up. But we're happy to help with your refund.",
    validation_summaries=[
        ValidationSummary(
            validator_name='CompetitorCheck',
            validator_status='fail',
            property_path='$',
            failure_reason='Found the following competitors: PayPal, Stripe. Please avoid naming those competitors next time',
            error_spans=[
                ErrorSpan(
                    start=10,
                    end=16,
                    reason='Competitor was found: Stripe'
                ),
                ErrorSpan(
                    start=45,
                    end=51,
                    reason='Competitor was found: PayPal'
                )
            ]
        )
    ],
    validated_output="Thanks for reaching out!Honestly, [COMPETITOR] has lower fees than 

ERROR:opentelemetry.exporter.otlp.proto.http.trace_exporter:Failed to export span batch due to timeout, max retries or shutdown.


### 2. FILTER OnFailAction

In [ ]:
guard = competitor_guard(OnFailAction.FILTER)
out = guard.validate(BAD_TEXT)
print("Output Raw response", out)
print("validation_passed:", out.validation_passed)
print("OUTPUT:\n", out.validated_output)

Output Raw response ValidationOutcome[TypeVar](
    call_id='4828317744',
    raw_llm_output="Thanks for reaching out! Honestly, Stripe has lower fees than us, and PayPal is easier to set up. But we're happy to help with your refund.",
    validation_summaries=[
        ValidationSummary(
            validator_name='CompetitorCheck',
            validator_status='fail',
            property_path='$',
            failure_reason='Found the following competitors: PayPal, Stripe. Please avoid naming those competitors next time',
            error_spans=[
                ErrorSpan(
                    start=10,
                    end=16,
                    reason='Competitor was found: Stripe'
                ),
                ErrorSpan(
                    start=45,
                    end=51,
                    reason='Competitor was found: PayPal'
                )
            ]
        )
    ],
    validated_output=None,
    reask=None,
    validation_passed=False,
    error=None
)


ERROR:opentelemetry.exporter.otlp.proto.http.trace_exporter:Failed to export span batch due to timeout, max retries or shutdown.


### 3. REFRAIN OnFailAction

In [ ]:
guard = competitor_guard(OnFailAction.REFRAIN)
out = guard.validate(BAD_TEXT)
print("Output Raw response", out)
print("validation_passed:", out.validation_passed)
print("OUTPUT:\n", out.validated_output)

Output Raw response ValidationOutcome[TypeVar](
    call_id='4827420320',
    raw_llm_output="Thanks for reaching out! Honestly, Stripe has lower fees than us, and PayPal is easier to set up. But we're happy to help with your refund.",
    validation_summaries=[
        ValidationSummary(
            validator_name='CompetitorCheck',
            validator_status='fail',
            property_path='$',
            failure_reason='Found the following competitors: PayPal, Stripe. Please avoid naming those competitors next time',
            error_spans=[
                ErrorSpan(
                    start=10,
                    end=16,
                    reason='Competitor was found: Stripe'
                ),
                ErrorSpan(
                    start=45,
                    end=51,
                    reason='Competitor was found: PayPal'
                )
            ]
        )
    ],
    validated_output=None,
    reask=None,
    validation_passed=False,
    error=None
)


ERROR:opentelemetry.exporter.otlp.proto.http.trace_exporter:Failed to export span batch due to timeout, max retries or shutdown.


### 4. REFRAIN vs FILTER

In [10]:
from typing import Optional
from pydantic import BaseModel, Field

STRUCTURED_JSON = (
    '{"answer": "Refunds settle within 24 hours.",'
    ' "comparison": "Honestly Stripe is cheaper than us.",'   # <- this field names a competitor
    ' "category": "refund"}'
)

In [11]:
def structured_guard(action):
    cc = CompetitorCheck(competitors=COMPETITORS, on_fail=action, use_local=True)
    class SupportReply(BaseModel):
        answer: str
        comparison: Optional[str] = Field(default=None, json_schema_extra={"validators": [cc]})
        category: str
    return Guard.for_pydantic(SupportReply)

In [ ]:
for action in (OnFailAction.FILTER, OnFailAction.REFRAIN):
    guard = structured_guard(action)
    guard.parse(STRUCTURED_JSON)
    it = guard.history[-1].iterations[-1]
    print(f"=== {action} ===")
    print("  after parsing (sentinel placed in the bad field):")
    print("   ", it.parsed_output)
    print("  guarded_output (what the guard actually returns):")
    print("   ", it.guarded_output)
    print("/n")

/Users/rishabh013/prod_new/guardrails/gaurdrails_ai/.venv/lib/python3.14/site-packages/guardrails/validator_service/__init__.py:81: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


=== OnFailAction.FILTER ===
  after parsing (sentinel placed in the bad field):
    {'answer': 'Refunds settle within 24 hours.', 'comparison': <guardrails.actions.filter.Filter object at 0x120de6710>, 'category': 'refund'}
  guarded_output (what the guard actually returns):
    {'answer': 'Refunds settle within 24 hours.', 'category': 'refund'}
/n


=== OnFailAction.REFRAIN ===
  after parsing (sentinel placed in the bad field):
    {'answer': 'Refunds settle within 24 hours.', 'comparison': <guardrails.actions.refrain.Refrain object at 0x123314a50>, 'category': 'refund'}
  guarded_output (what the guard actually returns):
    {}
/n


ERROR:opentelemetry.exporter.otlp.proto.http.trace_exporter:Failed to export span batch due to timeout, max retries or shutdown.
ERROR:opentelemetry.exporter.otlp.proto.http.trace_exporter:Failed to export span batch due to timeout, max retries or shutdown.


### 5. EXCEPTION OnFailAction

In [ ]:
guard = competitor_guard(OnFailAction.EXCEPTION)
try:
    out = guard.validate(BAD_TEXT)
    print("Raw Output", out)
    print("Passed (unexpected):", out.validated_output)
except ValidationError as e:
    print("Error Raw Output", out)
    print("Raised ValidationError")
    print(str(e)[:300])

Error Raw Output ValidationOutcome[TypeVar](
    call_id='4827420320',
    raw_llm_output="Thanks for reaching out! Honestly, Stripe has lower fees than us, and PayPal is easier to set up. But we're happy to help with your refund.",
    validation_summaries=[
        ValidationSummary(
            validator_name='CompetitorCheck',
            validator_status='fail',
            property_path='$',
            failure_reason='Found the following competitors: PayPal, Stripe. Please avoid naming those competitors next time',
            error_spans=[
                ErrorSpan(
                    start=10,
                    end=16,
                    reason='Competitor was found: Stripe'
                ),
                ErrorSpan(
                    start=45,
                    end=51,
                    reason='Competitor was found: PayPal'
                )
            ]
        )
    ],
    validated_output=None,
    reask=None,
    validation_passed=False,
    error=None
)
Rai

ERROR:opentelemetry.exporter.otlp.proto.http.trace_exporter:Failed to export span batch due to timeout, max retries or shutdown.


### 6. ReASK OnFailAction

In [15]:
BOT_MODEL = "gpt-4o-mini"
SYSTEM_PROMPT = (
    "You are NimbusPay's customer-support assistant. "
    "NimbusPay is a digital payments app: cards, wallets, refunds, account help. "
    "Be concise, friendly, and accurate."
)

In [ ]:

reask_guard = Guard().use(
    CompetitorCheck(competitors=COMPETITORS, on_fail=OnFailAction.REASK, use_local=True)
)
res = reask_guard(
    model = BOT_MODEL,
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": (
            "A customer asks why they should pick NimbusPay. In your answer, explicitly "
            "compare us to Stripe and PayPal by name."
        )},
    ],
    temperature = 0.3,
    num_reasks=2,
)

/Users/rishabh013/prod_new/guardrails/gaurdrails_ai/.venv/lib/python3.14/site-packages/guardrails/validator_service/__init__.py:81: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


ERROR:opentelemetry.exporter.otlp.proto.http.trace_exporter:Failed to export span batch due to timeout, max retries or shutdown.


In [ ]:
print("FINAL OUTPUT:\n", res.validated_output)

FINAL OUTPUT:
 Great question! Here’s why you might choose NimbusPay:

1. **User-Friendly Interface**: NimbusPay offers a more intuitive and streamlined experience, making it easier for users to navigate and manage their payments.

2. **Lower Fees**: We pride ourselves on competitive transaction fees, which can save you money in the long run.

3. **Integrated Wallet**: NimbusPay combines card payments and digital wallets seamlessly, allowing for quick and easy transactions without juggling multiple platforms.

4. **Customer Support**: Our dedicated customer support team is available to assist you promptly, ensuring you get the help you need when you need it.

5. **Flexible Refunds**: We provide a straightforward refund process that is easier to manage compared to the more complex systems of other payment solutions.

Overall, NimbusPay is designed to be a hassle-free solution for your digital payment needs!


ERROR:opentelemetry.exporter.otlp.proto.http.trace_exporter:Failed to export span batch due to timeout, max retries or shutdown.


In [18]:
print(res)

ValidationOutcome[TypeVar](
    call_id='4965469072',
    raw_llm_output='Great question! Here’s why you might choose NimbusPay:\n\n1. **User-Friendly Interface**: NimbusPay offers a more intuitive and streamlined experience, making it easier for users to navigate and manage their payments.\n\n2. **Lower Fees**: We pride ourselves on competitive transaction fees, which can save you money in the long run.\n\n3. **Integrated Wallet**: NimbusPay combines card payments and digital wallets seamlessly, allowing for quick and easy transactions without juggling multiple platforms.\n\n4. **Customer Support**: Our dedicated customer support team is available to assist you promptly, ensuring you get the help you need when you need it.\n\n5. **Flexible Refunds**: We provide a straightforward refund process that is easier to manage compared to the more complex systems of other payment solutions.\n\nOverall, NimbusPay is designed to be a hassle-free solution for your digital payment needs!',
    val

In [19]:
iters = reask_guard.history[-1].iterations
print("\nLLM round-trips:", len(iters), "| Reask occurred:", len(iters) > 1)

# Show what the model actually returned on EACH attempt (raw, pre-validation):
for i, it in enumerate(iters, start=1):
    print(f"\n--- Attempt {i} (status: {it.status}) ---")
    print(repr(it.raw_output))


LLM round-trips: 2 | Reask occurred: True

--- Attempt 1 (status: fail) ---
'Great question! Here’s why you might choose NimbusPay over Stripe or PayPal:\n\n1. **User-Friendly Interface**: NimbusPay offers a more intuitive and streamlined experience, making it easier for users to navigate and manage their payments.\n\n2. **Lower Fees**: We pride ourselves on competitive transaction fees, often lower than those of Stripe and PayPal, which can save you money in the long run.\n\n3. **Integrated Wallet**: NimbusPay combines card payments and digital wallets seamlessly, allowing for quick and easy transactions without juggling multiple platforms.\n\n4. **Customer Support**: Our dedicated customer support team is available to assist you promptly, ensuring you get the help you need when you need it, often faster than the support offered by Stripe and PayPal.\n\n5. **Flexible Refunds**: We provide a straightforward refund process that is easier to manage compared to the more complex systems o